# ESSAformer — Hyperspectral Super-Resolution Training

**ESSAformer** (ICCV 2023, transformer) for 6× EMIT→CNMF GT upsampling.

1. Setup & Drive
2. Data copy
3. Training
4. Evaluation

## 1. Setup

In [ ]:
import wandb
wandb.login()

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, textwrap
os.environ['GH_USER'] = userdata.get('GH_USER')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')

askpass_path = '/tmp/gh_askpass.sh'
with open(askpass_path, 'w') as f:
    f.write(textwrap.dedent('''\
        #!/bin/sh\n\
        case \"$1\" in\n\
          *Username*) echo \"$GH_USER\" ;;\n\
          *Password*) echo \"$GH_TOKEN\" ;;\n\
          *) echo \"\" ;;\n\
        esac\n\
    '''))
os.chmod(askpass_path, 0o700)
os.environ['GIT_ASKPASS'] = askpass_path
os.environ['GIT_TERMINAL_PROMPT'] = '0'

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git 2>/dev/null || echo 'Repo already cloned'

In [ ]:
!pip install rasterio wandb pyyaml tqdm -q

## 2. Data

In [ ]:
import shutil
from pathlib import Path

src = Path('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-04-02/zips_cnmf')
dst = Path('/content/data/zips_cnmf')

if not dst.exists():
    print(f'Copying {sum(1 for _ in src.glob("*.zip"))} zips to local disk...')
    shutil.copytree(src, dst)
    print('Done')
else:
    print(f'Data already copied ({sum(1 for _ in dst.glob("*.zip"))} zips)')

In [ ]:
%cd /content/HyperSpectralSuperResolution/training
!git pull

## 3. Training

In [ ]:
!python train.py \
    --config configs/essaformer_colab.yaml

In [ ]:
# Resume from checkpoint (uncomment if resuming)
# !python train.py \
#     --config configs/essaformer_colab.yaml \
#     --resume /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/experiments/essaformer-L1/models/iter_50000.pth

## 4. Evaluation

In [ ]:
!python evaluate.py \
    --config configs/essaformer_colab.yaml \
    --checkpoint /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/experiments/essaformer-L1/models/best.pth \
    --n-vis 8

In [ ]:
from IPython.display import display, Image
from pathlib import Path

fig_dir = Path('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/experiments/essaformer-L1/eval_test/figures')
for png in sorted(fig_dir.glob('*_main.png'))[:4]:
    print(png.name)
    display(Image(filename=str(png), width=800))